# 🧠 Train Python, C, and Java Autocomplete LLM on GPU (Google Colab / Kaggle)

This notebook trains a custom **33M parameter GPT-style decoder-only Transformer** from scratch for Python, C, and Java code completion.

### ⚡ Setup Instructions
1. In the top menu, go to **Runtime > Change runtime type**.
2. Select **GPU** (T4 GPU or P100 GPU are free and perfect for this task).
3. Run the cells sequentially.

## 1. Clone/Upload Codebase & Install Dependencies

In [ ]:
# Install required packages
!pip install tokenizers torch tqdm matplotlib

# If you have this codebase zipped, upload it to Colab and unzip it:
# !unzip Code-AutoComplete-LLM.zip -d .
# %cd Code-AutoComplete-LLM

## 2. Download and Extract Multilingual Datasets
This downloads the clean Python, C, and Java algorithm datasets.

In [ ]:
!python tools/download_dataset.py

## 3. Preprocess and Clean Source Files
This filters out test files, strips large docstrings, removes duplicates, and compiles the code into standard training and validation files.

In [ ]:
!python tools/hardened_clean.py
!python tools/build_train_file.py

## 4. Train the Custom Byte-Pair Encoding (BPE) Tokenizer
This trains the BPE tokenizer with a vocabulary of 8000 tokens on our Python, C, and Java corpus.

In [ ]:
!python tokenizer/train_tokenizer.py

## 5. Verify Dataset and Model Statistics

In [ ]:
!python tools/inspect_dataset.py
!python tools/model_stats.py

## 6. Start LLM Training
We will run the training loop. We can specify training parameters like epochs, learning rate, and batch size here.

In [ ]:
!python training/train.py --epochs 3 --batch_size 16 --grad_accum 4 --lr 3e-4

## 7. Plot Training Loss
Visualize the training loss curve step-by-step.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import csv

steps = []
losses = []

with open("training_logs/loss_log.csv", "r") as f:
    reader = csv.reader(f)
    for row in reader:
        steps.append(int(row[0]))
        losses.append(float(row[1]))

plt.figure(figsize=(10, 6))
plt.plot(steps, losses)
plt.xlabel("Training Step")
plt.ylabel("Cross Entropy Loss")
plt.title("Multilingual LLM Training Loss Curve")
plt.grid(True)
plt.show()

## 8. Run Inference & Testing
Test your model by typing prompts in Python, C, or Java!

In [ ]:
# Example Python Autocomplete
!python inference/run_model.py -c model/checkpoints/latest_checkpoint.pth -p "def dfs(graph, node, visited):"

# Example C Autocomplete
!python inference/run_model.py -c model/checkpoints/latest_checkpoint.pth -p "int fibonacci(n) {"

# Example Java Autocomplete
!python inference/run_model.py -c model/checkpoints/latest_checkpoint.pth -p "public class Stack {"